# 📚 RAG: Teaching AI to Read Before Answering

**Chapter 06 — Notebook 1 of 2**

> *Retrieval-Augmented Generation: The open-book exam approach to AI*

## 🎓 What Even IS RAG?

### The Open-Book Exam Analogy

Imagine you're taking a history test. You have three options:

```
Option A (Fine-tuning):
  📖📖📖  →  🧠 cram everything  →  ✍️ answer from memory
  You spent MONTHS memorizing. Works great — until new events happen!

Option B (Prompt Engineering):
  Teacher whispers: "Hint: the answer involves Napoleon" → ✍️ answer
  Quick and easy — but the teacher can only whisper so much!

Option C (RAG = Open-Book Exam):
  📚 Bring your textbooks  →  🔍 look up relevant pages  →  ✍️ write answer
  You combine your reasoning skills WITH up-to-date references. Best of both worlds!
```

### Real Products That Use RAG

| Product | What It Does | Why RAG? |
|---------|-------------|----------|
| **Perplexity.ai** | Answers questions with cited web sources | News is updated daily — can't fine-tune that fast! |
| **ChatPDF** | Lets you "chat" with any PDF | Your private PDF wasn't in the training data |
| **GitHub Copilot (workspace)** | Answers about YOUR codebase | Your repo is private and unique |
| **Notion AI** | Answers from your Notion docs | Personal knowledge base = private data |
| **Customer Support Bots** | Answers from company knowledge bases | Company docs change frequently |

### The Core Idea in One Sentence

> **RAG = Search for relevant documents FIRST, then pass them to an LLM to generate an answer.**

The LLM doesn't need to "memorize" your documents — it just reads the relevant snippets at inference time, like looking something up in a book! 📖

## ⚔️ Finetuning vs Prompt Engineering vs RAG

These three approaches answer the same question: *"How do we give an LLM the knowledge it needs?"*

### Comparison Table

| Dimension | 🏋️ Fine-Tuning | 💬 Prompt Engineering | 📚 RAG |
|-----------|--------------|----------------------|--------|
| **How it works** | Retrain model weights on your data | Stuff context into the prompt | Search docs → inject top results → generate |
| **Knowledge update** | Requires retraining (expensive!) | Update prompt manually | Just update the document store |
| **Handles private data** | Yes (after training) | Yes (if fits in context) | Yes (at query time) |
| **Up-to-date info** | ❌ Stale after cutoff | ✅ If you inject it | ✅ Search live data |
| **Cost** | 💰💰💰 High (compute) | 💰 Low | 💰💰 Medium (storage + retrieval) |
| **Latency** | Fast at inference | Fast at inference | Adds retrieval step (~100–500ms) |
| **Hallucination risk** | Medium | Medium–High | Lower (grounded in docs) |
| **Interpretability** | ❌ Hard to audit | ✅ Can see what's in prompt | ✅ Can show source docs |
| **Best for** | Style/format changes, domain adaptation | Simple context injection, few-shot | Large knowledge bases, dynamic data |
| **Worst for** | Frequently changing info | Large private corpora (context limits) | Tasks needing deep reasoning without external refs |

### When to Use What?

```
❓ Question: "What is our company's refund policy?"

  Fine-tuning:  Train the model on the policy doc. But what when policy changes??
  
  Prompt Eng:   Paste the policy in every prompt. But policy is 50 pages long!!
  
  RAG:          Store policy docs → retrieve relevant section → answer. ✅ WINNER

❓ Question: "Write me a tweet in our brand voice"

  Fine-tuning:  Train on past tweets to learn brand voice. ✅ WINNER
  
  Prompt Eng:   Add 3 example tweets to every prompt. Also works!
  
  RAG:          Overkill — retrieving "brand voice docs" is unnecessary complexity.
```

> **Staff-level insight:** In practice, these aren't mutually exclusive! Many production systems combine fine-tuning (for style/behavior) WITH RAG (for knowledge). The best system design depends on your data freshness requirements, corpus size, and latency budget.

In [ ]:
# 🧪 Cell 4: Conceptual demonstration — what each approach actually sends to the LLM
# (No real LLM calls — just showing the STRUCTURE of what gets sent)

user_question = "What were Q3 2024 revenue figures for Acme Corp?"

# ─────────────────────────────────────────────────────────────────────────────
# APPROACH 1: Fine-tuning
# The model was trained on Acme Corp's documents. So the prompt is just... the question.
# The knowledge is BAKED INTO the weights.
# ─────────────────────────────────────────────────────────────────────────────
finetuned_prompt = f"""
[SYSTEM]: You are Acme Corp's financial assistant.
(The model was fine-tuned on all Acme Corp earnings reports)

[USER]: {user_question}
"""

# ─────────────────────────────────────────────────────────────────────────────
# APPROACH 2: Prompt Engineering
# We manually paste the relevant doc section into every prompt.
# Problem: what if the earnings report is 200 pages?
# ─────────────────────────────────────────────────────────────────────────────
manually_injected_context = """
[Manually pasted from earnings report — hoping we picked the right section!]
Q3 2024 Financial Highlights: Total Revenue $2.3B (+12% YoY)...
...Operating Income $420M...Free Cash Flow $180M...
(This might be wrong section, or miss other relevant details!)
"""
prompt_eng_prompt = f"""
[SYSTEM]: You are a financial assistant. Use the context below.

[CONTEXT]: {manually_injected_context}

[USER]: {user_question}
"""

# ─────────────────────────────────────────────────────────────────────────────
# APPROACH 3: RAG
# The system SEARCHES a document store, retrieves the most relevant chunks,
# and dynamically injects them. This happens automatically at query time!
# ─────────────────────────────────────────────────────────────────────────────
retrieved_chunks = [
    "[Chunk from Q3_2024_earnings.pdf, page 3]: Q3 2024 Revenue: $2.3B (+12% YoY)",
    "[Chunk from Q3_2024_earnings.pdf, page 5]: Q3 2024 Operating Income: $420M",
    "[Chunk from investor_call_transcript.pdf]: CEO noted Q3 beat guidance by 4%",
]

rag_prompt = f"""
[SYSTEM]: You are a financial assistant. Answer ONLY using the retrieved context below.
          If the context doesn't contain the answer, say "I don't know."

[RETRIEVED CONTEXT]:
{chr(10).join(f'  {i+1}. {chunk}' for i, chunk in enumerate(retrieved_chunks))}

[USER]: {user_question}
"""

# ─────────────────────────────────────────────────────────────────────────────
# Print comparison
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("APPROACH 1: Fine-Tuning")
print("=" * 70)
print(finetuned_prompt)
print("👆 Clean prompt! But knowledge is frozen at training time.")

print("\n" + "=" * 70)
print("APPROACH 2: Prompt Engineering")
print("=" * 70)
print(prompt_eng_prompt)
print("👆 Context manually pasted. Fragile for large/changing docs.")

print("\n" + "=" * 70)
print("APPROACH 3: RAG (Automatically Retrieved Context)")
print("=" * 70)
print(rag_prompt)
print("👆 Context auto-retrieved! Scales to millions of docs. Citable sources.")

## 📄 Step 1 of RAG: Document Parsing

Before we can search documents, we need to extract their text. Sounds simple — but documents are MESSY!

```
Raw Document (PDF/HTML/DOCX)
          │
          ▼
   ┌─────────────┐
   │   PARSER    │  ← The hard part!
   └─────────────┘
          │
          ▼
   Clean Text Chunks
```

### Two Main Approaches

#### 🔧 Rule-Based Parsing
Uses explicit rules like: "text between `<p>` tags is a paragraph", "bold text in PDFs is probably a header"

- **Tools:** PyPDF2, pdfminer, python-docx, BeautifulSoup
- **Pros:** Fast, cheap, deterministic, no GPU needed
- **Cons:** Breaks on complex layouts (multi-column, tables, figures)

```
Example — simple HTML rule:
  <h1>Revenue Report</h1>   →  "Revenue Report" [type: heading]
  <p>Q3 was great...</p>   →  "Q3 was great..." [type: paragraph]
  <table>...</table>        →  ??? (tables are notoriously hard!)
```

#### 🤖 AI-Based Parsing (LayoutParser)
Uses computer vision + NLP to understand document STRUCTURE, not just text.

**LayoutParser Pipeline Steps:**

```
Step 1: Document Image / PDF
           │
           ▼
Step 2: Layout Detection (Object Detection Model)
        Detects regions: [Title] [Text] [Figure] [Table] [List]
        ┌─────────────────────────────────┐
        │  ┌──────────────────┐           │
        │  │  [TITLE REGION]  │           │
        │  └──────────────────┘           │
        │  ┌──────────┐  ┌───────────┐   │
        │  │ [TEXT]   │  │ [FIGURE]  │   │
        │  │          │  │    📊     │   │
        │  └──────────┘  └───────────┘   │
        │  ┌──────────────────────────┐  │
        │  │       [TABLE]            │  │
        │  └──────────────────────────┘  │
        └─────────────────────────────────┘
           │
           ▼
Step 3: OCR per Region (Tesseract / PaddleOCR)
        Extract text from each detected region separately
           │
           ▼
Step 4: Reading Order Recovery
        Sort regions left→right, top→bottom (or by detected order)
           │
           ▼
Step 5: Structured Output
        [{type: "title", text: "Revenue Report"},
         {type: "text",  text: "Q3 was great..."},
         {type: "table", text: "| Q1 | Q2 | Q3 |
|...|"}]
```

**AI-based pros:** Handles complex layouts, figures, multi-column text, scanned PDFs  
**AI-based cons:** Needs GPU, slower, more complex setup

> **Staff-level insight:** For production RAG systems, the parsing step is often where most failures happen. A two-column academic PDF parsed naively produces garbled, interleaved text from both columns. Always validate your parser output on a sample of your actual documents before building the rest of the pipeline!

## ✂️ Step 2 of RAG: Document Chunking

Once we have clean text, we need to split it into **chunks** — small pieces that can be:
1. Individually embedded as vectors
2. Retrieved when relevant
3. Fit into the LLM's context window

```
Full Document (100,000 tokens)
         │
         ▼  Chunking
 [chunk1] [chunk2] [chunk3] ... [chunkN]
 (500 tok) (500 tok) (500 tok)    (500 tok)
         │
         ▼  Each chunk gets embedded
 [vec1]  [vec2]  [vec3]  ...  [vecN]
```

### Chunking Strategies

#### 1. 📏 Length-Based (Fixed Size)
Split every N characters or tokens, with optional overlap.

```python
# Split every 500 chars, with 50-char overlap
chunks = [text[i:i+500] for i in range(0, len(text), 450)]
```

- **Pros:** Simple, predictable chunk sizes
- **Cons:** Splits mid-sentence, breaks semantic coherence
- **When to use:** When semantic boundaries don't matter much (e.g., code)

#### 2. 🔤 Regex / Sentence-Based
Split on sentence boundaries (`[.!?]`) or paragraph breaks (`\n\n`).

```python
import re
sentences = re.split(r'(?<=[.!?])\s+', text)  # split on sentence endings
# Then group sentences until chunk_size is reached
```

- **Pros:** Respects natural language boundaries
- **Cons:** Sentence lengths vary wildly; some sentences are 3 words, some are 80
- **When to use:** News articles, general prose

#### 3. 📝 Markdown/HTML Splitters
Split on semantic structure markers: `#` headers, `<section>` tags, `---` dividers.

```
# Introduction          ← SPLIT HERE
This is intro text...

## Background           ← SPLIT HERE
More context...

### Key Concept         ← SPLIT HERE
Detailed explanation...
```

- **Pros:** Chunks are semantically coherent (each covers one topic/section)
- **Cons:** Requires structured documents; sections can be too long or too short
- **When to use:** Documentation, wikis, structured reports

#### 4. 🧠 Semantic / Agentic Chunking (Advanced)
Use an LLM or embedding model to detect topic shifts. Group sentences that are semantically similar.

```
Compute embeddings for each sentence
    │
    ▼
Detect sudden jumps in cosine similarity between adjacent sentences
    │                                        │
    ▼                                        ▼
  same topic = keep in same chunk     different topic = new chunk
```

- **Pros:** Best semantic coherence
- **Cons:** Expensive (needs embedding model per sentence), non-deterministic

### The Overlap Trick 🔁

```
chunk 1: |████████████████████░░░░░|
chunk 2:              |░░░░░████████████████████░░░░░|
chunk 3:                           |░░░░░████████████████████|

░ = overlap region (same text appears in both adjacent chunks)
```

Overlap ensures that if the answer spans a chunk boundary, at least one chunk will contain both parts of the relevant context.

> **Staff-level insight:** Chunk size is one of the most impactful hyperparameters in a RAG system. Too small → each chunk lacks context. Too large → you retrieve irrelevant content alongside relevant parts. A common starting point is 256–512 tokens with ~10–15% overlap. Always tune this empirically on your specific data!

In [ ]:
# 🧪 Cell 7: Text Chunking Implementation
# Configurable chunk_size and overlap, with examples on a real paragraph

import re
from typing import List


def chunk_by_length(text: str, chunk_size: int = 200, overlap: int = 40) -> List[str]:
    """Fixed-length chunking with overlap."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:  # skip empty chunks
            chunks.append(chunk)
        if end >= len(text):
            break
    return chunks


def chunk_by_sentences(text: str, max_chunk_size: int = 300, overlap_sentences: int = 1) -> List[str]:
    """Sentence-boundary chunking with sentence-level overlap."""
    # Split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    current_chunk_sentences = []
    current_length = 0
    
    for sentence in sentences:
        # If adding this sentence would exceed limit, flush current chunk
        if current_length + len(sentence) > max_chunk_size and current_chunk_sentences:
            chunks.append(" ".join(current_chunk_sentences))
            # Keep last `overlap_sentences` sentences for overlap
            current_chunk_sentences = current_chunk_sentences[-overlap_sentences:]
            current_length = sum(len(s) for s in current_chunk_sentences)
        
        current_chunk_sentences.append(sentence)
        current_length += len(sentence)
    
    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))
    
    return chunks


def chunk_by_markdown_headers(text: str) -> List[dict]:
    """Split on Markdown headers, returning dict with title + content."""
    # Split on lines that start with #
    pattern = re.compile(r'(?m)^(#{1,6}\s+.+)$')
    parts = pattern.split(text)
    
    chunks = []
    # parts alternates: [pre-header-text, header, content, header, content, ...]
    # Handle leading text before first header
    if parts[0].strip():
        chunks.append({"header": "[Introduction]", "content": parts[0].strip()})
    
    for i in range(1, len(parts) - 1, 2):
        header = parts[i].strip()
        content = parts[i + 1].strip() if (i + 1) < len(parts) else ""
        if content:
            chunks.append({"header": header, "content": content})
    
    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# SAMPLE TEXT
# ─────────────────────────────────────────────────────────────────────────────
sample_text = """\
Retrieval-Augmented Generation, commonly known as RAG, is a technique that enhances \
large language models by providing them with relevant external information at inference \
time. Unlike fine-tuning, which permanently modifies model weights, RAG leaves the base \
model unchanged. Instead, it adds a retrieval step before generation. The system first \
searches a document store for passages relevant to the user's query. Those passages are \
then injected into the prompt as context. Finally, the LLM generates an answer grounded \
in the retrieved documents. This approach has several key advantages. It allows the \
knowledge base to be updated without retraining. It provides source attribution, making \
answers auditable. And it reduces hallucinations by anchoring the model to factual text.\
"""

markdown_text = """\
# Introduction to RAG
RAG stands for Retrieval-Augmented Generation. It enhances LLMs with external knowledge.

## How Retrieval Works
The system embeds queries and documents into vector space. Nearest neighbor search finds relevant docs.

## How Generation Works
Retrieved docs are injected into the prompt. The LLM generates a grounded answer.

## Evaluation
We measure context relevance, faithfulness, and answer correctness.
"""

# ─────────────────────────────────────────────────────────────────────────────
# Run all three chunkers
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("METHOD 1: Fixed-Length Chunking (chunk_size=200, overlap=40)")
print("=" * 65)
length_chunks = chunk_by_length(sample_text, chunk_size=200, overlap=40)
for i, chunk in enumerate(length_chunks):
    print(f"  Chunk {i+1} ({len(chunk)} chars): '{chunk[:80]}...'")

print()
print("Overlap demonstration:")
if len(length_chunks) >= 2:
    # Show where chunks 1 and 2 overlap
    overlap_text = length_chunks[0][-40:]
    print(f"  End of chunk 1: '...{overlap_text}'")
    print(f"  Start of chunk 2: '{length_chunks[1][:40]}...'")
    overlap_found = length_chunks[1].startswith(overlap_text[:20])
    print(f"  Overlap detected: {overlap_found} ✅")

print()
print("=" * 65)
print("METHOD 2: Sentence-Based Chunking (max_chunk_size=300, overlap=1 sentence)")
print("=" * 65)
sentence_chunks = chunk_by_sentences(sample_text, max_chunk_size=300, overlap_sentences=1)
for i, chunk in enumerate(sentence_chunks):
    print(f"  Chunk {i+1} ({len(chunk)} chars): '{chunk[:90]}...'")

print()
print("=" * 65)
print("METHOD 3: Markdown Header Chunking")
print("=" * 65)
md_chunks = chunk_by_markdown_headers(markdown_text)
for i, chunk in enumerate(md_chunks):
    print(f"  Chunk {i+1}: Header='{chunk['header']}' | Content='{chunk['content'][:60]}...'")

print()
print("💡 Key takeaway: Sentence and markdown chunking respect semantic boundaries,")
print("   while fixed-length chunking can cut mid-sentence (but is simpler to implement).")

## 🔢 Step 3 of RAG: Vector Embeddings

After chunking, we need to convert text into numbers so we can do math on them.

### How a Text Encoder Works

```
Text chunk: "RAG retrieves documents before generating answers"
    │
    ▼  Text Encoder (e.g., sentence-transformers, OpenAI ada-002)
    │
    ▼
Dense Vector: [0.12, -0.87, 0.34, 0.09, -0.21, ..., 0.55]  ← 768 or 1536 dimensions!
```

### Why Dense Vectors are Amazing 🤯

The encoder is trained so that **semantically similar texts produce similar vectors**.

```
"RAG retrieves documents"    →  [0.12, -0.87, 0.34, ...]
"retrieval augmented gen"    →  [0.11, -0.89, 0.36, ...]  ← similar!
"how to make pizza"          →  [0.78,  0.32, -0.91, ...]  ← very different!

Vector space intuition:
                  🍕 pizza          dog 🐶
                                    🐕
                                   cat
  RAG 📚 ← close → retrieval
```

### Key Properties of Good Embeddings

| Property | Description | Example |
|----------|-------------|--------|
| **Semantic similarity** | Similar meaning → similar vectors | "car" ≈ "automobile" |
| **Compositionality** | Phrase meaning > sum of word meanings | "hot dog" ≠ "hot" + "dog" |
| **Cross-lingual** (some models) | Same meaning in different languages → similar vectors | "chat" (FR) ≈ "cat" (EN) |
| **Asymmetric search** | Query embedding vs. document embedding can use different encoders | bi-encoders |

### Popular Embedding Models

| Model | Dimensions | Best For |
|-------|-----------|----------|
| `text-embedding-ada-002` (OpenAI) | 1536 | General purpose, high quality |
| `text-embedding-3-small` (OpenAI) | 1536 | Cheaper, still great |
| `all-MiniLM-L6-v2` (HuggingFace) | 384 | Fast, free, good for prototypes |
| `BGE-large` (BAAI) | 1024 | State-of-the-art open source |
| `E5-large` (Microsoft) | 1024 | Instruction-tuned, very strong |

> **Staff-level insight:** The embedding model is one of the most important components to tune in a RAG system. Domain-specific models (e.g., a biomedical embedding model for medical RAG) can dramatically outperform general-purpose models. Also note that embedding dimensions are a cost-accuracy tradeoff — smaller dimensions = faster search + less storage, but lower quality.

In [ ]:
# 🧪 Cell 9: Simple Text Embeddings Using Character N-Grams
# No heavy model needed — demonstrates the CONCEPT that similar texts → similar vectors

import math
from collections import Counter
from typing import Dict, Tuple


def get_char_ngrams(text: str, n: int = 3) -> Counter:
    """Extract character n-grams from text (lowercased, spaces removed)."""
    clean = text.lower().replace(" ", "_")  # replace spaces with underscore
    return Counter(clean[i:i+n] for i in range(len(clean) - n + 1))


def ngram_to_vector(text: str, vocabulary: list, n: int = 3) -> list:
    """Convert text to a fixed-size vector using n-gram frequencies."""
    ngrams = get_char_ngrams(text, n)
    total = sum(ngrams.values()) or 1  # avoid division by zero
    # Each dimension = normalized frequency of that n-gram in vocabulary
    vector = [ngrams.get(gram, 0) / total for gram in vocabulary]
    return vector


def cosine_similarity(vec_a: list, vec_b: list) -> float:
    """Cosine similarity between two vectors. Range: [-1, 1]."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a ** 2 for a in vec_a))
    norm_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


# ─────────────────────────────────────────────────────────────────────────────
# Build a vocabulary from all our sample texts
# ─────────────────────────────────────────────────────────────────────────────
texts = [
    "retrieval augmented generation",     # RAG
    "vector search and document retrieval",  # Similar to RAG
    "embedding vectors for text similarity",  # Somewhat related
    "machine learning neural networks",   # ML but not RAG
    "how to bake chocolate chip cookies",  # Completely unrelated!
    "recipe for homemade bread and pastry",  # Also unrelated
]

# Build shared vocabulary from top-k n-grams across all texts
all_ngrams = Counter()
for t in texts:
    all_ngrams.update(get_char_ngrams(t, n=3))
vocabulary = [gram for gram, _ in all_ngrams.most_common(100)]  # top 100 n-grams

print(f"Vocabulary size: {len(vocabulary)} 3-grams")
print(f"Sample vocab: {vocabulary[:15]}")

# ─────────────────────────────────────────────────────────────────────────────
# Embed all texts
# ─────────────────────────────────────────────────────────────────────────────
embeddings = {text: ngram_to_vector(text, vocabulary) for text in texts}

print(f"\nEach embedding is a vector of {len(vocabulary)} dimensions")
print(f"Sample embedding for '{texts[0][:30]}...':")
sample_vec = embeddings[texts[0]]
print(f"  First 10 dims: {[round(v, 3) for v in sample_vec[:10]]}")
print(f"  Non-zero dims: {sum(1 for v in sample_vec if v > 0)} out of {len(sample_vec)}")

# ─────────────────────────────────────────────────────────────────────────────
# Compute pairwise similarities — show that similar texts have similar vectors
# ─────────────────────────────────────────────────────────────────────────────
query = texts[0]  # "retrieval augmented generation"
print(f"\n{'='*65}")
print(f"Query: '{query}'")
print(f"{'='*65}")
print(f"{'Text':<45} {'Cosine Sim':>12} {'Verdict':>10}")
print("-" * 70)

query_vec = embeddings[query]
results = []
for text in texts:
    sim = cosine_similarity(query_vec, embeddings[text])
    results.append((sim, text))

results.sort(reverse=True)
for sim, text in results:
    display_text = text[:43] + ".." if len(text) > 43 else text
    if text == query:
        verdict = "(self)"
    elif sim > 0.5:
        verdict = "✅ similar"
    elif sim > 0.2:
        verdict = "🟡 somewhat"
    else:
        verdict = "❌ different"
    print(f"  {display_text:<45} {sim:>10.3f}  {verdict}")

print()
print("💡 Even with simple character n-grams (no neural network!):")
print("   - RAG-related texts cluster together with higher similarity scores")
print("   - Cooking texts are far from RAG texts")
print("   Real embeddings (sentence-transformers, OpenAI) work MUCH better")
print("   because they capture MEANING, not just surface character patterns!")

In [ ]:
# 🧪 Cell 10: Nearest Neighbor Search — Brute Force Exact Search
# Implement cosine similarity search over a small set of document embeddings

import math
import time
from typing import List, Tuple


class ExactVectorStore:
    """Brute-force exact nearest neighbor search using cosine similarity.
    
    Time complexity: O(N * D) per query, where N = # docs, D = dimensions.
    This is the gold standard for accuracy but doesn't scale to millions of docs!
    """
    
    def __init__(self):
        self.documents: List[str] = []
        self.embeddings: List[List[float]] = []
    
    def add(self, document: str, embedding: List[float]):
        """Add a document and its embedding to the store."""
        self.documents.append(document)
        self.embeddings.append(embedding)
    
    def _cosine_similarity(self, vec_a: List[float], vec_b: List[float]) -> float:
        """Compute cosine similarity between two vectors."""
        dot = sum(a * b for a, b in zip(vec_a, vec_b))
        norm_a = math.sqrt(sum(a ** 2 for a in vec_a))
        norm_b = math.sqrt(sum(b ** 2 for b in vec_b))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)
    
    def search(self, query_embedding: List[float], top_k: int = 3) -> List[Tuple[float, str]]:
        """Brute-force search: compare query to ALL documents."""
        start_time = time.time()
        
        # Compute similarity with every single document — O(N * D) !
        scores = [
            (self._cosine_similarity(query_embedding, emb), doc)
            for emb, doc in zip(self.embeddings, self.documents)
        ]
        
        # Sort by similarity, return top_k
        scores.sort(key=lambda x: x[0], reverse=True)
        
        elapsed_ms = (time.time() - start_time) * 1000
        print(f"  [Exact search] Compared {len(self.documents)} docs in {elapsed_ms:.2f}ms")
        
        return scores[:top_k]


# ─────────────────────────────────────────────────────────────────────────────
# Build a small document store from our sample texts
# ─────────────────────────────────────────────────────────────────────────────
document_corpus = [
    "RAG stands for Retrieval-Augmented Generation, combining search with LLMs.",
    "Vector embeddings represent text as dense numerical vectors in high-dimensional space.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors.",
    "Fine-tuning updates model weights on domain-specific data to improve performance.",
    "HNSW is a graph-based approximate nearest neighbor algorithm with excellent performance.",
    "Document chunking splits long texts into smaller pieces for embedding and retrieval.",
    "Python is a versatile programming language popular in data science and AI.",
    "Cosine similarity measures the angle between two vectors, ignoring their magnitudes.",
    "Prompt engineering involves crafting effective instructions for language models.",
    "K-means clustering partitions data into k groups based on feature similarity.",
]

# Build vocabulary from the corpus
corpus_all_ngrams = Counter()
for doc in document_corpus:
    corpus_all_ngrams.update(get_char_ngrams(doc, n=3))
corpus_vocab = [gram for gram, _ in corpus_all_ngrams.most_common(200)]

# Create the vector store and add documents
store = ExactVectorStore()
for doc in document_corpus:
    emb = ngram_to_vector(doc, corpus_vocab)
    store.add(doc, emb)

print(f"Vector store loaded with {len(store.documents)} documents")
print(f"Each document is a {len(corpus_vocab)}-dimensional vector")

# ─────────────────────────────────────────────────────────────────────────────
# Run some queries
# ─────────────────────────────────────────────────────────────────────────────
queries = [
    "how does retrieval work in RAG systems",
    "algorithms for fast vector search",
    "how to split documents into chunks",
]

for query in queries:
    query_emb = ngram_to_vector(query, corpus_vocab)
    print(f"\n{'='*65}")
    print(f"🔍 Query: '{query}'")
    print(f"{'='*65}")
    results = store.search(query_emb, top_k=3)
    for rank, (score, doc) in enumerate(results, 1):
        print(f"  Rank {rank} (score={score:.3f}): {doc[:70]}")

print()
print("⚠️  Brute-force search works fine for 10 docs.")
print("   But at 1,000,000 docs, comparing every single vector is too slow!")
print("   That's why we need Approximate Nearest Neighbor (ANN) algorithms. 👇")

## 🚀 ANN Algorithms: Fast (Approximate) Search at Scale

Brute-force exact search is too slow for millions of documents. **Approximate Nearest Neighbor (ANN)** algorithms trade a tiny bit of accuracy for massive speed gains.

```
Exact Search:
  Query  →  compare with ALL 10M docs  →  top-k results
  Time: O(N × D)   ← gets slow fast!

ANN Search:
  Query  →  compare with ~1000 candidate docs (smart selection!)  →  top-k results
  Time: O(log N × D)  ← much faster!
  Accuracy: 95–99% recall (you might miss 1–5% of true top-k, acceptable!)
```

### ANN Algorithm Families

#### 🌲 1. Tree-Based (KD-Tree, Ball-Tree, Annoy)
```
Build phase: Recursively split vector space with hyperplanes
                    ROOT
                   /    \
                  /      \
              [left]    [right]   ← split on dimension with most variance
              /   \     /   \
            [LL] [LR] [RL] [RR]  ← keep splitting recursively

Query phase:  Traverse tree → find the leaf node your query belongs to
              Only compare vectors in that leaf!

✅ Great for low dimensions (<50D)
❌ Breaks down in high dimensions ("curse of dimensionality")
   768D embeddings → every vector is equidistant from the query!
```

#### #️⃣ 2. Locality Sensitive Hashing (LSH)
```
Key idea: Hash similar vectors to the SAME bucket

  vec_A = [0.1, 0.9, 0.2]  → hash → bucket_42
  vec_B = [0.1, 0.8, 0.3]  → hash → bucket_42  ← same bucket! (similar vectors)
  vec_C = [0.9, 0.1, 0.7]  → hash → bucket_17  ← different bucket

Query phase: Hash the query → only search in same bucket!

✅ Very fast, works well in high dimensions
❌ Tuning hash functions is tricky; can have false negatives
```

#### 🔵 3. Clustering-Based (IVF — Inverted File Index)
```
Build phase:
  1. Run k-means on all vectors → get k cluster centroids
  2. Assign each vector to its nearest centroid
  3. Store: centroid → list of vectors in that cluster

     centroid_1 ●  ← cluster 1 (nearby docs)
     centroid_2 ●  ← cluster 2 (nearby docs)
     centroid_3 ●  ← cluster 3 (nearby docs)

Query phase:
  1. Find nearest centroid(s) to query vector
  2. Search only within those clusters (nprobe=1 or more)
  3. Return top-k from searched clusters

✅ FAISS uses this! Scales to billions of vectors
❌ Requires training k-means (slow build, one-time cost)
```

#### 🕸️ 4. Graph-Based (HNSW — Hierarchical Navigable Small World)
```
The CURRENT state-of-the-art! Used in Pinecone, Weaviate, Qdrant.

Build phase: Create a multi-layer graph
  Layer 2 (few nodes, long-range connections):  A────────E
  Layer 1 (more nodes, medium connections):     A──B──C──D──E
  Layer 0 (all nodes, short-range connections): A─a─b─B─c─d─C─e─f─D─g─h─E

Query phase:
  1. Enter at top layer → quickly navigate to approximate region (big jumps)
  2. Drop to lower layers → refine with finer-grained navigation (small steps)
  3. At bottom layer → return top-k neighbors

  Like navigating a city: first take highways (Layer 2),
  then main roads (Layer 1), then local streets (Layer 0)!

✅ Best recall/speed tradeoff, handles high dimensions well
✅ No training required (incremental inserts supported!)
❌ High memory usage (stores graph structure)
❌ Build is O(N log N) — slower than IVF for very large corpora
```

### Algorithm Comparison

| Algorithm | Build Speed | Query Speed | Recall | Memory | Best Use Case |
|-----------|------------|-------------|--------|--------|---------------|
| Exact Search | O(1) | O(N·D) | 100% | Low | <100K docs, ground truth |
| KD-Tree | O(N log N) | O(log N) | ~100% | Low | Low-dim data |
| LSH | O(N) | O(1)–O(N) | ~90% | Medium | Very large, approximate ok |
| IVF (FAISS) | O(N·k) | O(k·n/K) | ~95% | Low-Med | Billions of vectors |
| HNSW | O(N log N) | O(log N) | ~99% | High | Most production RAG systems |

> **Staff-level insight:** HNSW is almost always the right default for RAG in 2024–2025. Its recall at 99% with sub-millisecond latency is hard to beat. If you're memory-constrained, use IVF+PQ (product quantization) which compresses vectors from 4 bytes/dim to 1 byte/dim with some accuracy loss.

In [ ]:
# 🧪 Cell 12: Clustering-Based ANN (IVF-style) — Compare to Exact Search
# Implement k-means clustering, then search only within nearest cluster(s)

import math
import random
import time
from typing import List, Tuple

random.seed(42)


def vector_add(a: List[float], b: List[float]) -> List[float]:
    return [x + y for x, y in zip(a, b)]

def vector_scale(v: List[float], s: float) -> List[float]:
    return [x * s for x in v]

def euclidean_distance(a: List[float], b: List[float]) -> float:
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

def cosine_sim(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x**2 for x in a))
    nb = math.sqrt(sum(x**2 for x in b))
    return dot / (na * nb) if na > 0 and nb > 0 else 0.0


class KMeansIndex:
    """Simplified IVF (Inverted File Index) using k-means clustering.
    
    This mirrors how FAISS's IndexIVFFlat works:
    1. Build: k-means cluster all vectors
    2. Search: find nearest centroid(s), search only those clusters
    """
    
    def __init__(self, n_clusters: int = 3, max_iter: int = 20):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids: List[List[float]] = []
        self.cluster_docs: dict = {}   # cluster_id → list of (embedding, doc_text)
        self.total_docs = 0
    
    def _assign_clusters(self, embeddings: List[List[float]]) -> List[int]:
        """Assign each embedding to nearest centroid."""
        assignments = []
        for emb in embeddings:
            nearest = min(
                range(len(self.centroids)),
                key=lambda i: euclidean_distance(emb, self.centroids[i])
            )
            assignments.append(nearest)
        return assignments
    
    def build(self, documents: List[str], embeddings: List[List[float]]):
        """Run k-means and build the inverted index."""
        self.total_docs = len(documents)
        dim = len(embeddings[0])
        
        # Initialize centroids by random sampling (k-means++  simplified)
        self.centroids = [embeddings[i] for i in random.sample(range(len(embeddings)), self.n_clusters)]
        
        # Iterate: assign → recompute centroids
        for iteration in range(self.max_iter):
            assignments = self._assign_clusters(embeddings)
            
            # Recompute centroids as mean of assigned embeddings
            new_centroids = []
            for c in range(self.n_clusters):
                cluster_embs = [emb for emb, a in zip(embeddings, assignments) if a == c]
                if cluster_embs:
                    mean = vector_scale(
                        [sum(dim_vals) for dim_vals in zip(*cluster_embs)],
                        1.0 / len(cluster_embs)
                    )
                    new_centroids.append(mean)
                else:
                    new_centroids.append(self.centroids[c])  # keep old centroid if empty
            
            # Check convergence
            if all(euclidean_distance(old, new) < 1e-6
                   for old, new in zip(self.centroids, new_centroids)):
                break
            self.centroids = new_centroids
        
        # Build inverted index
        self.cluster_docs = {c: [] for c in range(self.n_clusters)}
        for doc, emb, cluster_id in zip(documents, embeddings, assignments):
            self.cluster_docs[cluster_id].append((emb, doc))
        
        # Print cluster stats
        print(f"  K-means converged. Cluster sizes: ", end="")
        for c in range(self.n_clusters):
            print(f"  Cluster {c}: {len(self.cluster_docs[c])} docs", end="")
        print()
    
    def search(self, query_emb: List[float], top_k: int = 3, nprobe: int = 1) -> List[Tuple[float, str]]:
        """ANN search: find nprobe nearest clusters, search only those.
        
        nprobe=1: search only the nearest cluster (fastest, lowest recall)
        nprobe=n_clusters: equivalent to exact search (slowest, highest recall)
        """
        start_time = time.time()
        
        # Step 1: Find nearest nprobe centroids
        centroid_distances = [
            (euclidean_distance(query_emb, c), i)
            for i, c in enumerate(self.centroids)
        ]
        centroid_distances.sort()
        nearest_cluster_ids = [cid for _, cid in centroid_distances[:nprobe]]
        
        # Step 2: Search only within selected clusters
        candidates = []
        docs_searched = 0
        for cid in nearest_cluster_ids:
            for emb, doc in self.cluster_docs[cid]:
                candidates.append((cosine_sim(query_emb, emb), doc))
                docs_searched += 1
        
        candidates.sort(reverse=True)
        
        elapsed_ms = (time.time() - start_time) * 1000
        print(f"  [ANN search, nprobe={nprobe}] Searched {docs_searched}/{self.total_docs} docs "
              f"({100*docs_searched/self.total_docs:.0f}%) in {elapsed_ms:.3f}ms")
        
        return candidates[:top_k]


# ─────────────────────────────────────────────────────────────────────────────
# Build embeddings for the corpus
# ─────────────────────────────────────────────────────────────────────────────
all_embs = [ngram_to_vector(doc, corpus_vocab) for doc in document_corpus]

# Build the ANN index
print("Building K-Means ANN index...")
ann_index = KMeansIndex(n_clusters=3, max_iter=50)
ann_index.build(document_corpus, all_embs)

# Also build exact search store for comparison
exact_store = ExactVectorStore()
for doc, emb in zip(document_corpus, all_embs):
    exact_store.add(doc, emb)

# ─────────────────────────────────────────────────────────────────────────────
# Compare ANN vs Exact Search
# ─────────────────────────────────────────────────────────────────────────────
test_queries = [
    "fast similarity search for dense vectors",
    "how to split documents for RAG",
]

for query in test_queries:
    q_emb = ngram_to_vector(query, corpus_vocab)
    
    print(f"\n{'='*65}")
    print(f"🔍 Query: '{query}'")
    print(f"{'='*65}")
    
    print("EXACT SEARCH:")
    exact_results = exact_store.search(q_emb, top_k=3)
    exact_top1 = exact_results[0][1] if exact_results else None
    for rank, (score, doc) in enumerate(exact_results, 1):
        print(f"   Rank {rank} ({score:.3f}): {doc[:65]}")
    
    print("\nANN SEARCH (nprobe=1, 1 cluster only):")
    ann_results_1 = ann_index.search(q_emb, top_k=3, nprobe=1)
    ann_top1_1 = ann_results_1[0][1] if ann_results_1 else None
    for rank, (score, doc) in enumerate(ann_results_1, 1):
        print(f"   Rank {rank} ({score:.3f}): {doc[:65]}")
    print(f"   Top-1 match: {'✅ Same as exact' if ann_top1_1 == exact_top1 else '⚠️ Different from exact'}")
    
    print("\nANN SEARCH (nprobe=2, 2 clusters):")
    ann_results_2 = ann_index.search(q_emb, top_k=3, nprobe=2)
    ann_top1_2 = ann_results_2[0][1] if ann_results_2 else None
    for rank, (score, doc) in enumerate(ann_results_2, 1):
        print(f"   Rank {rank} ({score:.3f}): {doc[:65]}")
    print(f"   Top-1 match: {'✅ Same as exact' if ann_top1_2 == exact_top1 else '⚠️ Different from exact'}")

print()
print("Key insight: nprobe controls the recall/speed tradeoff!")
print("  nprobe=1  → fastest, might miss some results (lower recall)")
print("  nprobe=k  → same as exact search, slowest")
print("  nprobe=2  → good middle ground for most applications")

In [ ]:
# 🧪 Cell 13: FAISS — How You'd Use It in Production
# (Code shown as pseudocode/commented since faiss may not be installed)
# This shows the exact API you'd use in a real RAG system!

print("=" * 70)
print("FAISS: Facebook AI Similarity Search")
print("The most widely used vector search library in production RAG systems")
print("=" * 70)

faiss_example_code = '''
import faiss
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# SETUP: Your embeddings (N docs, each D dimensions)
# ─────────────────────────────────────────────────────────────────────────────
D = 768          # embedding dimension (e.g., sentence-transformers)
N = 1_000_000    # number of documents (1 million!)

# Embeddings must be float32 numpy arrays — FAISS requirement
embeddings = np.random.random((N, D)).astype('float32')

# ─────────────────────────────────────────────────────────────────────────────
# OPTION 1: Exact Search (IVF = 0 clusters)
# Good for < 100K docs
# ─────────────────────────────────────────────────────────────────────────────
index_exact = faiss.IndexFlatL2(D)       # L2 distance (Euclidean)
# OR: index_exact = faiss.IndexFlatIP(D)  # Inner product (use for cosine if normalized!)
index_exact.add(embeddings)             # O(N) build time

# ─────────────────────────────────────────────────────────────────────────────
# OPTION 2: IVF (Clustering-Based ANN)
# Good for 100K–10M docs
# ─────────────────────────────────────────────────────────────────────────────
n_clusters = 1024                       # rule of thumb: sqrt(N)
quantizer = faiss.IndexFlatL2(D)        # used to assign vectors to clusters
index_ivf = faiss.IndexIVFFlat(quantizer, D, n_clusters)
index_ivf.train(embeddings)             # runs k-means (one-time cost!)
index_ivf.add(embeddings)               # assign each vector to a cluster
index_ivf.nprobe = 8                    # search 8 nearest clusters (tune for recall/speed)

# ─────────────────────────────────────────────────────────────────────────────
# OPTION 3: IVF + PQ (Compressed, Memory-Efficient)
# Good for 10M+ docs where memory is a concern
# PQ = Product Quantization: compresses vectors from 4 bytes/dim to ~1 byte/dim
# ─────────────────────────────────────────────────────────────────────────────
M = 16           # number of sub-vectors (must divide D)
nbits = 8        # bits per sub-vector (256 centroids)
index_ivfpq = faiss.IndexIVFPQ(quantizer, D, n_clusters, M, nbits)
index_ivfpq.train(embeddings)
index_ivfpq.add(embeddings)
# Memory: 768D float32 = 3KB/vector → PQ reduces to ~16 bytes/vector = 200x compression!

# ─────────────────────────────────────────────────────────────────────────────
# OPTION 4: HNSW (Graph-Based, Best Recall)
# Best recall/speed tradeoff, but higher memory usage
# ─────────────────────────────────────────────────────────────────────────────
M_hnsw = 32      # number of connections per node (higher = better recall, more memory)
index_hnsw = faiss.IndexHNSWFlat(D, M_hnsw)
index_hnsw.hnsw.efConstruction = 40    # build quality (higher = better but slower)
index_hnsw.hnsw.efSearch = 16          # search quality (tune at query time!)
index_hnsw.add(embeddings)             # no training needed!

# ─────────────────────────────────────────────────────────────────────────────
# QUERYING (same API for all index types!)
# ─────────────────────────────────────────────────────────────────────────────
query_embedding = np.random.random((1, D)).astype('float32')  # your query vector
top_k = 5

distances, indices = index_ivf.search(query_embedding, top_k)
# distances: shape (1, top_k) — L2 distances to each result
# indices:   shape (1, top_k) — indices into your original embeddings array

for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1}: doc_idx={idx}, distance={dist:.4f}")
    # Use idx to retrieve the actual document text from your document store

# ─────────────────────────────────────────────────────────────────────────────
# SAVING / LOADING (important for production!)
# ─────────────────────────────────────────────────────────────────────────────
faiss.write_index(index_ivf, "my_index.faiss")  # save to disk
loaded_index = faiss.read_index("my_index.faiss")  # load from disk (fast!)
'''

print(faiss_example_code)

print("=" * 70)
print("FAISS Index Decision Guide:")
print("=" * 70)

decision_table = [
    ("< 100K docs",   "IndexFlatL2/IP",  "Exact",  "None",    "✅ Simple, accurate"),
    ("100K–5M docs",  "IndexIVFFlat",    "ANN",    "k-means", "⚡ Fast, good recall"),
    ("> 5M docs",     "IndexIVFPQ",      "ANN+PQ", "k-means", "💾 Memory efficient"),
    ("Any size",      "IndexHNSWFlat",   "Graph",  "None",    "🏆 Best recall/speed"),
]

print(f"  {'Scale':<15} {'Index Type':<18} {'Method':<8} {'Training':<10} {'Notes'}")
print("-" * 75)
for scale, idx_type, method, training, notes in decision_table:
    print(f"  {scale:<15} {idx_type:<18} {method:<8} {training:<10} {notes}")

print()
print("Alternative vector DBs that wrap FAISS / implement similar algorithms:")
print("  - Pinecone (managed cloud)")
print("  - Weaviate (open source, schema-based)")
print("  - Qdrant (open source, Rust-based, great performance)")
print("  - Chroma (lightweight, great for prototypes)")
print("  - pgvector (Postgres extension — SQL + vector search together!)")

## 🧠 Quiz Time! Retrieval Fundamentals

Test your knowledge! Try to answer before revealing the answers.

---

**Q1: What is the main advantage of RAG over fine-tuning for a customer support bot that needs to answer questions about your latest product documentation?**

<details>
<summary>👆 Click to reveal answer</summary>

RAG allows you to update the knowledge base (document store) without retraining the model. When your product docs change, you just re-embed and index the new docs — no expensive GPU training required. Fine-tuning would require a full retraining cycle every time docs change, which is too slow and expensive for frequently updated documentation.

</details>

---

**Q2: You're chunking a 200-page technical manual with chunk_size=500 tokens and overlap=50 tokens. Why do you include the overlap, and what does it prevent?**

<details>
<summary>👆 Click to reveal answer</summary>

Overlap prevents "boundary miss" problems. Without overlap, if the answer to a query spans the boundary between two chunks (e.g., the question context is at the end of chunk 5 and the answer is at the start of chunk 6), neither chunk alone contains the full relevant context. Overlap ensures adjacent chunks share some text, so at least one chunk will contain the full relevant passage. The 50-token overlap means each chunk shares its last 50 tokens with the beginning of the next chunk.

</details>

---

**Q3: What is the "curse of dimensionality" and why does it make KD-trees impractical for 768-dimensional embeddings?**

<details>
<summary>👆 Click to reveal answer</summary>

In high-dimensional spaces, the concept of "nearest neighbor" breaks down: all points become approximately equidistant from each other. For KD-trees, this means the spatial partitioning becomes meaningless — the tree degenerates to checking almost every node before finding the nearest neighbor, making it no better than brute force. With 768 dimensions (typical for sentence embeddings), KD-trees offer essentially zero speedup. HNSW and IVF handle high dimensions much better because they use different geometric intuitions (graph navigation and cluster assignment) that remain meaningful even at high dimensions.

</details>

---

**Q4: In FAISS's IVF index, what does `nprobe` control, and what is the tradeoff when you increase it?**

<details>
<summary>👆 Click to reveal answer</summary>

`nprobe` controls how many clusters are searched during a query. With `nprobe=1`, only the single nearest cluster is searched (fastest, lowest recall). With `nprobe=n_clusters`, all clusters are searched (equivalent to exact search, highest recall but slowest). Increasing nprobe improves recall (fewer missed relevant documents) at the cost of higher query latency. A common production choice is nprobe=8–16, which achieves ~95–99% recall with 5–10x speedup over exact search. This is the fundamental recall/speed tradeoff of ANN algorithms.

</details>

---

**Q5: Explain why HNSW (Hierarchical Navigable Small World) is called "hierarchical" and how that hierarchy helps speed up search.**

<details>
<summary>👆 Click to reveal answer</summary>

HNSW is hierarchical because it builds a multi-layer graph where higher layers contain fewer nodes with longer-range connections, and lower layers contain more nodes with shorter-range connections. This is analogous to a road network: highways (top layer) let you travel long distances quickly but stop infrequently; local streets (bottom layer) let you navigate precisely but are slow for long distances.

During search: you enter at the top layer and quickly navigate to the approximate region of the graph containing your query's nearest neighbors (big jumps, few comparisons). Then you drop to lower layers for increasingly fine-grained navigation. This greedy traversal achieves O(log N) query time instead of O(N) for brute force, while maintaining very high recall (~99%) because the multi-layer structure ensures you don't get trapped in local optima.

</details>